In [1]:
import torch
print(torch.cuda.is_available())

True


In [6]:
from datasets import load_dataset
dataset = load_dataset("ganler/code-r1-12k")

Generating test split: 100%|███████████████████████████████████████████████████████████████████████████████████| 712/712 [00:00<00:00, 85935.50 examples/s]


In [8]:
print(dataset['train'][0])

{'task_id': None, 'prompt': [{'content': 'You are a helpful programming assistant. The user will ask you a question and you as the assistant solve it. The assistant first thinks how to solve the task through reasoning and then provides the user with the final answer. The reasoning process and answer are enclosed within <think>...</think> and <answer>...</answer> tags, respectively.', 'role': 'system'}, {'content': 'Solve the programming task below in a Python markdown code block.\nYou will have a list of rationals in the form\n\n```\nlst = [ [numer_1, denom_1] , ... , [numer_n, denom_n] ]\n```\nor\n```\nlst = [ (numer_1, denom_1) , ... , (numer_n, denom_n) ]\n```\n\nwhere all numbers are positive integers. You have to produce their sum `N / D` in an irreducible form: this means that `N` and `D` have only `1` as a common divisor.\n\nReturn the result in the form:\n\n- `[N, D]` in Ruby, Crystal, Python, Clojure, JS, CS, PHP, Julia\n- `Just "N D"` in Haskell, PureScript\n- `"[N, D]"` in J

In [35]:
import pandas as pd
df = pd.DataFrame(dataset['train'])
# display(df.head(1))
print(df.iloc[100]['prompt'])

[{'content': 'You are a helpful programming assistant. The user will ask you a question and you as the assistant solve it. The assistant first thinks how to solve the task through reasoning and then provides the user with the final answer. The reasoning process and answer are enclosed within <think>...</think> and <answer>...</answer> tags, respectively.', 'role': 'system'}, {'content': "Solve the programming task below in a Python markdown code block.\nFox Ciel is playing a card game with her friend Fox Jiro. There are n piles of cards on the table. And there is a positive integer on each card.\n\nThe players take turns and Ciel takes the first turn. In Ciel's turn she takes a card from the top of any non-empty pile, and in Jiro's turn he takes a card from the bottom of any non-empty pile. Each player wants to maximize the total sum of the cards he took. The game ends when all piles become empty.\n\nSuppose Ciel and Jiro play optimally, what is the score of the game?\n\nInput\n\nThe f

In [32]:
# Create prompt_id column, so we can group together the constraints
import ast

def make_prompt_id(extra_info):
    if isinstance(extra_info, str):
        extra_info = ast.literal_eval(extra_info)
    dataset = extra_info['dataset'].split('/')[-1]
    return f"{dataset}_{extra_info['index']}"

df['prompt_id'] = df['extra_info'].apply(make_prompt_id)

# Check if primary key
df['prompt_id'].is_unique

True

In [42]:
import copy

techniques = ['for-loop', 'while-loop', ...]

new_rows = []
for _, row in df.iterrows():
    # Keep the original row
    new_rows.append(row.copy())
    
    for i in range(1, len(techniques) + 1):
        new_row = row.copy()
        prompt = copy.deepcopy(row['prompt'])
        
        constraints = "\n".join(f"- {techniques[j-1]}" for j in range(1, i + 1))
        suffix = f"\nProgramming constraints: DO NOT use the following techniques\n{constraints}"
        
        content = prompt[1]['content']
        first_newline = content.index('\n')
        prompt[1]['content'] = content[:first_newline] + suffix + content[first_newline:]
        
        new_row['prompt'] = prompt
        new_rows.append(new_row)

expanded_df = pd.DataFrame(new_rows).reset_index(drop=True)

In [46]:
print(expanded_df.iloc[2]['prompt'][1])

{'content': 'Solve the programming task below in a Python markdown code block.\nProgramming constraints: DO NOT use the following techniques\n- for-loop\n- while-loop\nYou will have a list of rationals in the form\n\n```\nlst = [ [numer_1, denom_1] , ... , [numer_n, denom_n] ]\n```\nor\n```\nlst = [ (numer_1, denom_1) , ... , (numer_n, denom_n) ]\n```\n\nwhere all numbers are positive integers. You have to produce their sum `N / D` in an irreducible form: this means that `N` and `D` have only `1` as a common divisor.\n\nReturn the result in the form:\n\n- `[N, D]` in Ruby, Crystal, Python, Clojure, JS, CS, PHP, Julia\n- `Just "N D"` in Haskell, PureScript\n- `"[N, D]"` in Java, CSharp, TS, Scala, PowerShell, Kotlin\n- `"N/D"` in Go, Nim\n- `{N, D}` in C++, Elixir\n- `{N, D}` in C\n- `Some((N, D))` in Rust\n- `Some "N D"` in F#, Ocaml\n- `c(N, D)` in R\n- `(N, D)` in Swift\n- `\'(N D)` in Racket\n\nIf the result is an integer (`D` evenly divides `N`) return:\n\n- an integer in Ruby, Cry